In [ ]:
import matplotlib.pyplot as plt

from src.core.config import get_model_configs
from src.core.paths import (
    PAPER_STYLE_FILENAME,
    PLOTS_DIR,
    get_run_dirs,
)
from src.core.utils import (
    DifferenceVector,
    ModelName,
    PoolingType,
    Q1Statistic,
    SplitSuffixes,
)
from src.plots.data_loaders import (
    load_pair_edit_data,
    load_q1_projections,
    load_q1_results,
    load_q2_results,
    load_q3_results,
    load_q4_results,
    load_q4_robustness_results,
    select_q1_results,
)
from src.plots.plot_settings import (
    COLUMN_WIDTH,
    REMOVED_CORPUS_NAMES,
)
from src.plots.plot_utils import plot_and_save
from src.plots.q1_plots import (
    plot_q1_forest,
    plot_q1_histogram,
    plot_q1_layer_sweep,
)
from src.plots.q2_plots import (
    plot_q2_baselines,
    plot_q2_by_definition,
    plot_q2_layer_sweep,
    plot_q2_primary,
)
from src.plots.q3_plots import (
    plot_q3_by_bin,
    plot_q3_levels,
)
from src.plots.q4_plots import (
    plot_q4_calibration,
    plot_q4_robustness,
    plot_q4_shares,
    plot_q4_strata,
    plot_q4_weights,
)
from src.plots.tables import (
    write_q1_table,
    write_q2_table,
    write_q3_table,
    write_q4_table,
    write_replication_table,
)


_, experiment_config = get_model_configs(ModelName.LLAMA_3B)
run_dirs = get_run_dirs(experiment_config)
corpus_name, split, layer = 'real_aae', SplitSuffixes.TESTING, experiment_config.primary_layer
results_dir, figures_dir = run_dirs.pair_results, run_dirs.figures
plt.style.use(PLOTS_DIR / f'{PAPER_STYLE_FILENAME}.mplstyle')

In [2]:
q1_results_list = load_q1_results(corpus_name, split, results_dir)
pair_edit_data = load_pair_edit_data(corpus_name, split, results_dir)
histogram_panels = [
    (
        load_q1_projections(corpus_name, split, layer, pooling_type, results_dir),
        *select_q1_results(q1_results_list, DifferenceVector.DIALECT, layer, pooling_type),
        pair_edit_data,
    )
    for pooling_type in PoolingType
]
plot_and_save(
    plot_q1_histogram,
    histogram_panels,
    'q1_histograms',
    figures_dir,
    height_ratio=0.4,
    sharey=True,
    legend_on_last=True,
    show_legend=False,
)
plot_and_save(
    plot_q1_forest,
    [(q1_results_list, statistic, experiment_config.q1.labeled_bounds, layer) for statistic in Q1Statistic],
    'q1_forest',
    figures_dir,
    height_ratio=0.45,
    sharey=True,
    legend_on_last=True,
)
plot_and_save(
    plot_q1_layer_sweep,
    [(q1_results_list, statistic) for statistic in Q1Statistic],
    'q1_layer_sweep',
    figures_dir,
    legend_on_last=True,
    primary_layer=layer,
)
write_q1_table(
    q1_results_list,
    layer,
    figures_dir,
)

In [3]:
q2_results_list = [
    results
    for results in load_q2_results(run_dirs.transfer_results)
    if results['corpus_name'] not in REMOVED_CORPUS_NAMES
]
q2_config = experiment_config.q2
corpus_names = tuple(
    corpus_name
    for corpus_name in q2_config.primary_corpus_names
    if corpus_name not in REMOVED_CORPUS_NAMES
)
pooling_types, min_auroc = q2_config.primary_pooling_types, q2_config.min_auroc
plot_and_save(
    plot_q2_primary,
    [(q2_results_list, corpus_names, pooling_types, min_auroc)],
    'q2_primary',
    figures_dir,
    width=COLUMN_WIDTH,
    height_ratio=0.8,
)
plot_and_save(
    plot_q2_baselines,
    [(q2_results_list, corpus_names, pooling_types[0])],
    'q2_baselines',
    figures_dir,
    width=COLUMN_WIDTH,
    height_ratio=0.7,
)
plot_and_save(
    plot_q2_layer_sweep,
    [(q2_results_list, q2_config.midpoint_corpus_names[0], pooling_types, min_auroc, layer)],
    'q2_layer_sweep',
    figures_dir,
    width=COLUMN_WIDTH,
    height_ratio=0.7,
)

for q2_corpus_name in corpus_names:
    plot_and_save(
        plot_q2_by_definition,
        [(q2_results_list, q2_corpus_name, layer, pooling_types, min_auroc)],
        f'q2_by_definition_{q2_corpus_name}',
        figures_dir,
        width=COLUMN_WIDTH,
        height_ratio=0.9,
    )

write_q2_table(q2_results_list, figures_dir)

In [4]:
q3_results_list = load_q3_results(corpus_name, split, results_dir)
plot_and_save(
    plot_q3_levels,
    [(q3_results_list,)],
    'q3_levels',
    figures_dir,
    width=COLUMN_WIDTH,
    height_ratio=0.8,
)
plot_and_save(
    plot_q3_by_bin,
    [(q3_results,) for q3_results in q3_results_list],
    'q3_by_bin',
    figures_dir,
    sharey=True,
)
write_q3_table(q3_results_list, figures_dir)

In [5]:
q4_results_list = [
    load_q4_results(corpus_name, split, layer, pooling_type, results_dir)
    for pooling_type in experiment_config.q3_q4_pooling_types
]
q4_panels = [(q4_results,) for q4_results in q4_results_list]
plot_and_save(
    plot_q4_weights,
    q4_panels,
    'q4_weights',
    figures_dir,
    height_ratio=0.5,
    sharey=True,
    legend_on_last=True,
)
plot_and_save(
    plot_q4_shares,
    q4_panels,
    'q4_shares',
    figures_dir,
    height_ratio=0.5,
    sharey=True,
)
plot_and_save(
    plot_q4_strata,
    q4_panels,
    'q4_strata',
    figures_dir,
)
plot_and_save(
    plot_q4_calibration,
    q4_panels,
    'q4_calibration',
    figures_dir,
)

primary_q4_results, = [
    q4_results
    for q4_results in q4_results_list
    if q4_results['pooling_type'] == experiment_config.primary_pooling_type
]
robustness_results_list = load_q4_robustness_results(corpus_name, split, layer, experiment_config.primary_pooling_type, results_dir)
plot_and_save(
    plot_q4_robustness,
    [(robustness_results_list, primary_q4_results)],
    'q4_robustness',
    figures_dir,
    width=COLUMN_WIDTH,
    height_ratio=1.1,
)

for q4_results in q4_results_list:
    write_q4_table(q4_results, figures_dir)

In [ ]:
# generated
REPLICATION_MODELS = [
    (ModelName.LLAMA_3B, 'Llama-3.2-3B', '14 of 28'),
    (ModelName.LLAMA_8B, 'Llama-3.1-8B', '16 of 32'),
    (ModelName.OLMO_7B, 'OLMo-3 7B', '16 of 32'),
    (ModelName.GEMMA_2B, 'Gemma-2-2B', '13 of 26'),
]
replication_rows = []
for model_name, model_label, layer_label in REPLICATION_MODELS:
    _, model_experiment_config = get_model_configs(model_name)
    model_run_dirs = get_run_dirs(model_experiment_config)
    replication_rows.append((
        model_label,
        layer_label,
        model_experiment_config.primary_layer,
        load_q1_results(corpus_name, split, model_run_dirs.pair_results),
        load_q2_results(model_run_dirs.transfer_results),
    ))
write_replication_table(replication_rows, figures_dir)